# 03 — Model Training & Evaluation
**Forest Carbon Stock Estimation — Nainital District, Uttarakhand**

This notebook covers:
- Data loading and train/test split
- Hyperparameter tuning for XGBoost (RandomizedSearchCV)
- Random Forest baseline training
- Full evaluation: MAE, RMSE, R², residuals
- SHAP explainability (feature contributions per prediction)
- Cross-validation learning curves

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import joblib
from pathlib import Path

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import (train_test_split, KFold,
    cross_val_score, RandomizedSearchCV, learning_curve)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

with open('../config/config.yaml') as f:
    CFG = yaml.safe_load(f)
PATHS  = CFG['paths']
ML_CFG = CFG['ml']
FEAT   = CFG['features']

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120})
print('Setup complete.')

## 1. Load Data & Split

In [ ]:
df = pd.read_csv(Path('..') / PATHS['training_csv'])
print(f'Dataset: {len(df):,} samples × {len(df.columns)} columns')

features = FEAT['model_features']
target   = ML_CFG['target']

X = df[features].values
y = df[target].values

# Stratified split by AGBD quantile
strat = pd.qcut(y, q=5, labels=False, duplicates='drop')
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=ML_CFG['test_size'],
    random_state=ML_CFG['random_state'], stratify=strat
)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

## 2. XGBoost Hyperparameter Tuning (RandomizedSearchCV)

In [ ]:
from scipy.stats import randint, uniform

param_dist = {
    'n_estimators':     randint(200, 700),
    'max_depth':        randint(4, 10),
    'learning_rate':    uniform(0.01, 0.15),
    'subsample':        uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'min_child_weight': randint(1, 8),
    'reg_alpha':        uniform(0, 0.5),
    'reg_lambda':       uniform(0.5, 2.0),
}

base_xgb = XGBRegressor(
    eval_metric='rmse',
    random_state=ML_CFG['random_state'],
    n_jobs=-1
)

search = RandomizedSearchCV(
    base_xgb, param_dist,
    n_iter=30,          # increase to 60-100 for final run
    cv=5,
    scoring='r2',
    random_state=ML_CFG['random_state'],
    n_jobs=-1,
    verbose=1
)
search.fit(X_train, y_train)

print(f'\nBest R² (CV): {search.best_score_:.4f}')
print('Best params:')
for k, v in search.best_params_.items():
    print(f'  {k}: {v}')

## 3. Train Final Models

In [ ]:
# XGBoost — best params from search
xgb = XGBRegressor(
    **search.best_params_,
    eval_metric='rmse',
    early_stopping_rounds=30,
    random_state=ML_CFG['random_state'],
    n_jobs=-1
)
xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
xgb_pred = xgb.predict(X_test)

# Random Forest — baseline
rf = RandomForestRegressor(**ML_CFG['random_forest'],
                            random_state=ML_CFG['random_state'])
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

print('Models trained.')

## 4. Evaluation

In [ ]:
def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred)**0.5
    r2   = r2_score(y_true, y_pred)
    print(f'{name:20s}  MAE={mae:.2f}  RMSE={rmse:.2f}  R²={r2:.4f}')
    return dict(Model=name, MAE=mae, RMSE=rmse, R2=r2)

results = [
    evaluate('Random Forest', y_test, rf_pred),
    evaluate('XGBoost',       y_test, xgb_pred),
]
results_df = pd.DataFrame(results)
results_df

In [ ]:
# Scatter + residuals
fig, axes = plt.subplots(2, 2, figsize=(13, 11))
models = [('Random Forest', rf_pred), ('XGBoost', xgb_pred)]

for col, (name, preds) in enumerate(models):
    r2 = r2_score(y_test, preds)
    residuals = y_test - preds

    # Scatter
    ax = axes[0, col]
    ax.scatter(y_test, preds, alpha=0.3, s=10, color='#2e7d52', edgecolors='none')
    lim = [0, max(y_test.max(), preds.max()) + 10]
    ax.plot(lim, lim, 'r--', lw=1.5, label='1:1 line')
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel('Actual AGBD (t/ha)'); ax.set_ylabel('Predicted AGBD (t/ha)')
    ax.set_title(f'{name}  |  R²={r2:.3f}', fontweight='bold')
    ax.legend(fontsize=9); ax.spines[['top','right']].set_visible(False)

    # Residuals
    ax2 = axes[1, col]
    sns.histplot(residuals, bins=40, kde=True, ax=ax2, color='#4caf50')
    ax2.axvline(0, color='red', ls='--', lw=1.5)
    ax2.set_xlabel('Residual (Actual − Predicted, t/ha)')
    ax2.set_title(f'{name} Residuals', fontweight='bold')
    ax2.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.show()

## 5. SHAP Explainability (XGBoost)

In [ ]:
try:
    import shap
    explainer   = shap.TreeExplainer(xgb)
    shap_values = explainer.shap_values(X_test)

    # Summary beeswarm
    fig = plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_test, feature_names=features,
                      plot_type='dot', show=False)
    plt.title('SHAP Feature Contributions — XGBoost', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Bar summary
    fig2 = plt.figure(figsize=(9, 5))
    shap.summary_plot(shap_values, X_test, feature_names=features,
                      plot_type='bar', show=False)
    plt.title('Mean |SHAP| — Feature Importance', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

except ImportError:
    print('SHAP not installed. Run: pip install shap')

## 6. Learning Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
train_sizes = np.linspace(0.1, 1.0, 8)

for ax, (model, name) in zip(axes, [(rf, 'Random Forest'), (xgb, 'XGBoost')]):
    sizes, tr_scores, val_scores = learning_curve(
        model, X_train, y_train,
        train_sizes=train_sizes,
        cv=5, scoring='r2', n_jobs=-1
    )
    tr_mean  = tr_scores.mean(axis=1)
    val_mean = val_scores.mean(axis=1)
    tr_std   = tr_scores.std(axis=1)
    val_std  = val_scores.std(axis=1)

    ax.plot(sizes, tr_mean,  'o-', color='#1a5c38', label='Train R²')
    ax.plot(sizes, val_mean, 'o-', color='#e53935', label='CV R²')
    ax.fill_between(sizes, tr_mean-tr_std,   tr_mean+tr_std,   alpha=0.15, color='#1a5c38')
    ax.fill_between(sizes, val_mean-val_std, val_mean+val_std, alpha=0.15, color='#e53935')
    ax.axhline(0.70, ls='--', lw=1, color='gray', label='Target R²=0.70')
    ax.set_xlabel('Training samples'); ax.set_ylabel('R²')
    ax.set_title(f'Learning Curve — {name}', fontweight='bold')
    ax.legend(fontsize=9); ax.spines[['top','right']].set_visible(False)
    ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

## 7. Save Models

In [ ]:
model_dir = Path('..') / 'outputs' / 'models'
model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(rf,  model_dir / 'rf_model.joblib')
joblib.dump(xgb, model_dir / 'xgb_model.joblib')
print('Models saved to', model_dir)